In [1]:
import pandas as pd
from tqdm import tqdm
import json
from glob import glob
from collections import defaultdict
from pathlib import Path


In [6]:
config = json.load(open("../config/sdoh_extraction_schema_strict_type_simple_experiencer.json","r"))

In [20]:
output_schema = {}
for k,v in config.items():
    output_schema[k] = list(v['parameters']['properties']['extracted_conditions']['items']['properties'].keys())
    output_schema[k].remove("category")

In [21]:
output_schema

{'Healthcare': ['Experiencer', 'healthcare_type'],
 'Living': ['Experiencer', 'num_of_caregivers'],
 'Smoke': ['Experiencer', 'smoke_status'],
 'Employment': ['Experiencer', 'employment_status'],
 'Social': ['Experiencer', 'social_type', 'social_level'],
 'Education': ['Experiencer', 'education_status', 'education_level'],
 'Transportation': ['Experiencer',
  'transportation_type',
  'transportation_status'],
 'Mental Health': ['mentalhealth_type', 'Experiencer', 'mentalHealth_status'],
 'Insurance': ['Experiencer', 'insurance_type'],
 'Financial': ['Experiencer', 'financial_status'],
 'Substance Use': ['Experiencer', 'substanceuse_status'],
 'Trauma': ['Experiencer', 'trauma_type', 'trauma_status'],
 'Adherence': ['Experiencer', 'adherence_type', 'adherence_level'],
 'Literacy': ['Experiencer', 'literacy_type', 'literacy_level'],
 'Recommendation': ['Experiencer',
  'recommendation_type',
  'recommendation_status'],
 'Concern': ['concern_level']}

In [28]:
import json
import pandas as pd

# ==========================================
# 1. SCHEMAS AND MAPPINGS
# ==========================================

RAW_SCHEMA = {
    "Healthcare": ["Experiencer", "HealthcareType"],
    # "Living": ["Experiencer", "LivingStatus", "LivingType", "ResidentType"],
    "Living":["Experiencer"],
    "Smoke": ["Experiencer", "SmokeStatus"],
    "Employment": ["EmploymentStatus", "Experiencer"],
    "Social": ["Experiencer", "SocialActivity", "SocialType"],
    "Education": ["EducationStatus", "EducationType", "Experiencer"],
    "Transportation": ["Experiencer", "TransportationConvenienceLevel", "TransportationType"],
    "Mental Health": ["Experiencer", "MentalHealthStatus", "MentalHealthType"],
    "Insurance": ["Experiencer", "InsuranceType"],
    "Financial": ["Experiencer", "FinancialStatus"],
    "Substance Use": ["Experiencer", "SubstanceUseStatus"],
    "Trauma": ["Experiencer", "TraumaStatus", "TraumaType"],
    "Adherence": ["AdherenceLevel", "AdherenceType", "Experiencer"],
    "Literacy": ["Experiencer", "LiteracyLevel", "LiteracyType"],
    "Recommendation": ["RecommendationType"],
    "Concern": ["ConcernLevel"]
}

# Map the output keys to the ground truth keys (lowercase, no spaces, no underscores)
# Please change these values according to your specific annotation guidelines
KEY_MAPPING = {
    "sociallevel": "socialactivity",
    "transportationstatus": "transportationconveniencelevel",
    "educationlevel": "educationtype", 
    # "numofcaregivers": "livingstatus", # Or "residenttype", depending on your guidelines
    # "recommendationstatus": "recommendationtype"
}

NORMALIZED_SCHEMA = {}
for cat, keys in RAW_SCHEMA.items():
    # Remove both spaces and underscores for exact matching
    norm_cat = cat.lower().replace(" ", "").replace("_", "")
    norm_keys = sorted([k.lower().replace(" ", "").replace("_", "") for k in keys])
    NORMALIZED_SCHEMA[norm_cat] = norm_keys

# ==========================================
# 2. HELPER FUNCTIONS
# ==========================================
def map_experiencer(value):
    val = str(value).lower().strip()
    patient_group = ["patient", "the patient", "patients"]
    caregiver_group = ["father", "mother", "caregiver", "caregivers", 
                       "parents/caregiver", "family", "patients and caregivers", "parents"]
    if val in patient_group: return "patient"
    elif val in caregiver_group: return "caregivers"
    else: return "others"

def normalize_event_to_dict(event_dict):
    normalized_dict = {}
    for key, value in event_dict.items():
        # Standardize the key: lowercase, strip, remove spaces and underscores
        norm_key = key.lower().replace(" ", "").replace("_", "")
        if norm_key == "category": continue
        
        # Check if this key needs to be translated using our mapping
        if norm_key in KEY_MAPPING:
            norm_key = KEY_MAPPING[norm_key]
        
        # Standardize value: lowercase, strip, replace underscores with spaces
        norm_val = str(value).lower().strip().replace("_", " ")
        if norm_key == "experiencer":
            norm_val = map_experiencer(norm_val)
            
        normalized_dict[norm_key] = norm_val
    return normalized_dict

def dict_to_composite_tuple(category, event_dict):
    norm_cat = category.lower().replace(" ", "").replace("_", "")
    expected_keys = NORMALIZED_SCHEMA.get(norm_cat, sorted(list(event_dict.keys())))
    tuple_values = [event_dict.get(k, "not_mentioned") for k in expected_keys]
    return tuple(tuple_values)

# ==========================================
# 3. EVALUATION LOGIC
# ==========================================
def align_and_evaluate_composite_tuples(pred_events, true_events, doc_id, sentence, category):
    tp, fp, fn = 0, 0, 0
    records = []
    
    pred_dicts = [normalize_event_to_dict(e) for e in pred_events]
    true_dicts = [normalize_event_to_dict(e) for e in true_events]
    
    pred_tuples = [dict_to_composite_tuple(category, d) for d in pred_dicts]
    true_tuples = [dict_to_composite_tuple(category, d) for d in true_dicts]
    
    matched_true_indices = set()
    matched_pred_indices = set()
    
    # PASS 1: Greedy match based on >0 overlap
    for i, p_tuple in enumerate(pred_tuples):
        best_match_idx = -1
        best_overlap = 0
        
        for j, t_tuple in enumerate(true_tuples):
            if j in matched_true_indices: continue
            overlap = sum(1 for p_val, t_val in zip(p_tuple, t_tuple) if p_val == t_val)
            if overlap > best_overlap:
                best_overlap = overlap
                best_match_idx = j
                
        if best_match_idx != -1:
            matched_true_indices.add(best_match_idx)
            matched_pred_indices.add(i)
            t_tuple = true_tuples[best_match_idx]
            
            if best_overlap == len(p_tuple):
                tp += 1
                records.append([doc_id, sentence, category, t_tuple, p_tuple, "Exact Match (TP)"])
            else:
                fp += 1
                fn += 1
                records.append([doc_id, sentence, category, t_tuple, p_tuple, "Partial Match (FP/FN)"])

    # PASS 2: Force-pair remaining leftovers in same sentence/category (Overlap = 0)
    unmatched_preds = [(i, p) for i, p in enumerate(pred_tuples) if i not in matched_pred_indices]
    unmatched_trues = [(j, t) for j, t in enumerate(true_tuples) if j not in matched_true_indices]
    
    for (i, p_tuple), (j, t_tuple) in zip(unmatched_preds, unmatched_trues):
        matched_pred_indices.add(i)
        matched_true_indices.add(j)
        fp += 1
        fn += 1
        records.append([doc_id, sentence, category, t_tuple, p_tuple, "Complete Mismatch (FP/FN)"])

    # PASS 3: True Hallucinations 
    for i, p_tuple in enumerate(pred_tuples):
        if i not in matched_pred_indices:
            fp += 1
            records.append([doc_id, sentence, category, (), p_tuple, "Hallucination (FP)"])

    # PASS 4: True Misses 
    for j, t_tuple in enumerate(true_tuples):
        if j not in matched_true_indices:
            fn += 1
            records.append([doc_id, sentence, category, t_tuple, (), "Missed (FN)"])
            
    return tp, fp, fn, records

def evaluate_single_document(doc_id, true_filepath, pred_filepath):
    with open(true_filepath, 'r', encoding='utf-8') as f: true_data = json.load(f)
    with open(pred_filepath, 'r', encoding='utf-8') as f: pred_data = json.load(f)
        
    doc_category_counts = {}
    doc_records = []
    
    pred_lookup = {}
    for item in pred_data:
        sentence = item.get("sentence", "").strip()
        pred_lookup[sentence] = item.get("extracted_predictions", {})
        
    for idx, true_item in true_data.items():
        sentence = true_item.get("sentence", "").strip()
        true_sdoh_list = true_item.get("SDoH", [])
        pred_extracted = pred_lookup.get(sentence, {})
        
        all_categories = set()
        for event in true_sdoh_list: all_categories.update(event.keys())
        all_categories.update(pred_extracted.keys())
        
        for category in all_categories:
            true_events = [e[category] for e in true_sdoh_list if category in e]
            pred_events = pred_extracted.get(category, {}).get("extracted_conditions", [])
            
            tp, fp, fn, records = align_and_evaluate_composite_tuples(pred_events, true_events, doc_id, sentence, category)
            doc_records.extend(records)
            
            if category not in doc_category_counts:
                doc_category_counts[category] = {"tp": 0, "fp": 0, "fn": 0}
            doc_category_counts[category]["tp"] += tp
            doc_category_counts[category]["fp"] += fp
            doc_category_counts[category]["fn"] += fn
            
    return doc_category_counts, doc_records

def calculate_metrics(tp, fp, fn):
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0.0
    return precision, recall, f1

def evaluate_entire_corpus(document_file_pairs):
    corpus_category_metrics = {}
    all_corpus_records = []
    
    for doc_id, true_path, pred_path in document_file_pairs:
        doc_category_counts, doc_records = evaluate_single_document(doc_id, true_path, pred_path)
        all_corpus_records.extend(doc_records)
        
        for category, counts in doc_category_counts.items():
            if category not in corpus_category_metrics:
                corpus_category_metrics[category] = {"tp": 0, "fp": 0, "fn": 0}
            corpus_category_metrics[category]["tp"] += counts["tp"]
            corpus_category_metrics[category]["fp"] += counts["fp"]
            corpus_category_metrics[category]["fn"] += counts["fn"]

    print("=========================================")
    print("        OVERALL CORPUS METRICS           ")
    print("=========================================")
    total_tp, total_fp, total_fn = 0, 0, 0
    
    for category, counts in corpus_category_metrics.items():
        tp, fp, fn = counts["tp"], counts["fp"], counts["fn"]
        total_tp += tp
        total_fp += fp
        total_fn += fn
        c_p, c_r, c_f1 = calculate_metrics(tp, fp, fn)
        print(f"Category: {category}")
        print(f"  TP: {tp} | FP: {fp} | FN: {fn}  =>  F1: {c_f1:.4f}")

    print("\n--- GLOBAL SCORE ---")
    overall_p, overall_r, overall_f1 = calculate_metrics(total_tp, total_fp, total_fn)
    print(f"Overall Precision:  {overall_p:.4f}")
    print(f"Overall Recall:     {overall_r:.4f}")
    print(f"Overall F1 Score:   {overall_f1:.4f}")

    columns = ["doc_id", "sentence", "category", "true_label", "predicted_label", "status"]
    return pd.DataFrame(all_corpus_records, columns=columns)

In [29]:
# ==========================================
# 4. JUPYTER EXECUTION BLOCK
# ==========================================

# Using your exact file parsing logic
output_paths = sorted([Path(p).as_posix() for p in glob("../output/gpt55_type_simple_experiencer/*.json")])
true_paths = sorted([Path(p).as_posix() for p in glob("../data/processed_data/*.json")])

files = []
for i, j in zip(true_paths, output_paths):
    idx = i.rsplit("/", 1)[1].split("_")[0]
    files.append((idx, i, j))

print(f"Discovered {len(files)} document pairs. Starting evaluation...\n")

Discovered 33 document pairs. Starting evaluation...



In [30]:
# Run the evaluation pipeline
df_results = evaluate_entire_corpus(files)

# Save the detailed error analysis
output_csv_path = "../eval/sdoh_error_analysis.csv"
df_results.to_csv(output_csv_path, index=False)
print(f"\nSaved detailed error analysis to: {output_csv_path}")

# Preview the DataFrame in your notebook
df_results.head(15)

        OVERALL CORPUS METRICS           
Category: Healthcare
  TP: 118 | FP: 278 | FN: 244  =>  F1: 0.3113
Category: Living
  TP: 120 | FP: 67 | FN: 53  =>  F1: 0.6667
Category: Smoke
  TP: 24 | FP: 19 | FN: 20  =>  F1: 0.5517
Category: Employment
  TP: 117 | FP: 62 | FN: 55  =>  F1: 0.6667
Category: Social
  TP: 69 | FP: 108 | FN: 108  =>  F1: 0.3898
Category: Education
  TP: 34 | FP: 75 | FN: 72  =>  F1: 0.3163
Category: Transportation
  TP: 9 | FP: 140 | FN: 105  =>  F1: 0.0684
Category: Mental Health
  TP: 72 | FP: 86 | FN: 70  =>  F1: 0.4800
Category: Insurance
  TP: 41 | FP: 19 | FN: 23  =>  F1: 0.6613
Category: Financial
  TP: 74 | FP: 154 | FN: 165  =>  F1: 0.3169
Category: Substance Use
  TP: 50 | FP: 27 | FN: 28  =>  F1: 0.6452
Category: Trauma
  TP: 48 | FP: 74 | FN: 112  =>  F1: 0.3404
Category: Adherence
  TP: 36 | FP: 115 | FN: 110  =>  F1: 0.2424
Category: Literacy
  TP: 38 | FP: 67 | FN: 110  =>  F1: 0.3004
Category: Recommendation
  TP: 108 | FP: 41 | FN: 39  =>  F1:

,doc_id,sentence,category,true_label,predicted_label,status
0,100,"100\tPt, Tanecia, is a 4 year old female who i...",Healthcare,"(patient, diagnosis)","(patient, surgeries/procedures)",Partial Match (FP/FN)
1,100,"100\tPt, Tanecia, is a 4 year old female who i...",Healthcare,(),"(patient, hospital stay)",Hallucination (FP)
2,100,"100\tPt, Tanecia, is a 4 year old female who i...",Healthcare,(),"(patient, clinical visits)",Hallucination (FP)
3,100,"Living situation: Pt, Tanecia, lives with her...",Living,"(patient,)","(patient,)",Exact Match (TP)
4,100,"They reside at 4427 Dylan Loop, # 187, Land O ...",Living,"(caregivers,)","(patient,)",Complete Mismatch (FP/FN)
5,100,They reside in an apartment/ town home.,Living,"(caregivers,)","(patient,)",Complete Mismatch (FP/FN)
6,100,"There is no smoking in the home, although dad ...",Smoke,"(caregivers, none)","(patient, none)",Partial Match (FP/FN)
7,100,"There is no smoking in the home, although dad ...",Smoke,"(caregivers, current)","(caregivers, past)",Partial Match (FP/FN)
8,100,He was just diagnosed with asthma and is tryin...,Smoke,"(caregivers, current)","(caregivers, current)",Exact Match (TP)
9,100,Demographics: Mother: Courtney Hauss DOB: 0...,Employment,"(on leave, caregivers)","(on leave, caregivers)",Exact Match (TP)


In [25]:
output_paths = sorted([Path(p).as_posix() for p in glob("../output/gpt55/*.json")])
true_paths = sorted([Path(p).as_posix() for p in glob("../data/processed_data/*.json")])

In [26]:
files=[]
for i, j in zip(true_paths, output_paths):
    idx = i.rsplit("/",1)[1].split("_")[0]
    files.append((idx,i,j))

In [27]:
evaluate_entire_corpus(files)

        OVERALL CORPUS METRICS           
Category: Healthcare
  TP: 114 | FP: 264 | FN: 248  =>  F1: 0.3081
Category: Living
  TP: 0 | FP: 186 | FN: 173  =>  F1: 0.0000
Category: Smoke
  TP: 24 | FP: 19 | FN: 20  =>  F1: 0.5517
Category: Employment
  TP: 124 | FP: 52 | FN: 48  =>  F1: 0.7126
Category: Social
  TP: 1 | FP: 160 | FN: 176  =>  F1: 0.0059
Category: Education
  TP: 30 | FP: 81 | FN: 76  =>  F1: 0.2765
Category: Transportation
  TP: 0 | FP: 112 | FN: 114  =>  F1: 0.0000
Category: Mental Health
  TP: 59 | FP: 95 | FN: 83  =>  F1: 0.3986
Category: Insurance
  TP: 38 | FP: 13 | FN: 26  =>  F1: 0.6609
Category: Financial
  TP: 72 | FP: 170 | FN: 167  =>  F1: 0.2994
Category: Substance Use
  TP: 48 | FP: 28 | FN: 30  =>  F1: 0.6234
Category: Trauma
  TP: 2 | FP: 109 | FN: 158  =>  F1: 0.0148
Category: Adherence
  TP: 2 | FP: 139 | FN: 144  =>  F1: 0.0139
Category: Literacy
  TP: 0 | FP: 122 | FN: 148  =>  F1: 0.0000
Category: Recommendation
  TP: 1 | FP: 149 | FN: 146  =>  F1: 0

,doc_id,sentence,category,true_label,predicted_label,status
0,100,"100\tPt, Tanecia, is a 4 year old female who i...",Healthcare,"(patient, diagnosis)","(patient, surgeries/procedures)",Partial Match (FP/FN)
1,100,"100\tPt, Tanecia, is a 4 year old female who i...",Healthcare,(),"(patient, surgeries/procedures)",Hallucination (FP)
2,100,"100\tPt, Tanecia, is a 4 year old female who i...",Healthcare,(),"(patient, clinical visits)",Hallucination (FP)
3,100,"Living situation: Pt, Tanecia, lives with her...",Living,"(patient, current, with both parents, home)","(patient, 2)",Partial Match (FP/FN)
4,100,"They reside at 4427 Dylan Loop, # 187, Land O ...",Living,"(caregivers, current, home)","(caregivers, 2)",Partial Match (FP/FN)
...,...,...,...,...,...,...
2512,99,They also verbalize a commitment to the therap...,Healthcare,"(caregivers, clinical visits)","(caregivers, counseling)",Partial Match (FP/FN)
2513,99,Assessment and recommendations: Although t...,Literacy,"(caregivers, high, transplant knowledge)","(caregivers, not_mentioned, not_mentioned)",Partial Match (FP/FN)
2514,99,"However, they state they were advised during t...",Trauma,"(caregivers, past, loss)",(),Missed (FN)
2515,99,Recommendations are listed below: Recommen...,Recommendation,"(increase literacy,)","(not_mentioned,)",Complete Mismatch (FP/FN)
